In [1]:
# Install pip packages
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade --no-cache-dir "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn "torch==2.6" torchvision boto3

mkdir: cannot create directory ‘/home/ec2-user/tmpdir’: File exists
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.


In [8]:
# Prepare dataset
%cd ~/SageMaker/
! rm -rf dataset/ dataset.zip
!wget -nc -q -O "dataset.zip" "https://www.kaggle.com/api/v1/datasets/download/puneet6060/intel-image-classification"

!mkdir dataset -p
!unzip -q -o ./dataset.zip -d ./dataset/
!mv ./dataset/seg_train/seg_train ./dataset/train
!mv ./dataset/seg_test/seg_test ./dataset/test
!rm -r ./dataset/seg_* dataset.zip

%cd ./dataset/
!zip -rq ../dataset.zip .
%cd ..

/home/ec2-user/SageMaker
/home/ec2-user/SageMaker/dataset
/home/ec2-user/SageMaker


In [9]:
# Import pip packages
import pandas as pd
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from IPython.display import display
from sagemaker.inputs import TrainingInput
from sagemaker.s3 import S3Uploader

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [10]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [14]:
# Upload dataset zip file to S3
inputs = S3Uploader.upload("dataset.zip", "s3://{}/{}".format(bucket, prefix))

In [12]:
# Find mean, std
_dataset = datasets.ImageFolder(root='./dataset/train', transform=transforms.ToTensor())
_loader  = DataLoader(_dataset)

mean   = torch.zeros(3)
sq_mean = torch.zeros(3)
n_pixels = 0

with torch.no_grad():
    for images, _ in _loader:
        b, c, h, w = images.shape
        n_pixels += b * h * w
        mean     += images.sum(dim=[0, 2, 3])
        sq_mean  += (images ** 2).sum(dim=[0, 2, 3])

mean  /= n_pixels
std    = (sq_mean / n_pixels - mean ** 2).sqrt()

mean, std

(tensor([0.4302, 0.4575, 0.4538]), tensor([0.2694, 0.2679, 0.2983]))

In [15]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="train.py",
    framework_version="2.6",
    py_version="py312",
    instance_type="ml.g5.xlarge",
    instance_count=1,
    output_path="s3://{}/{}/output/".format(bucket, prefix),
    checkpoint_s3_uri="s3://{}/{}/checkpoint/".format(bucket, prefix),
    role=role,
    hyperparameters = {
        'epochs': 20,
        'batch_size': 64,
        'lr': 0.001,
        # 'backend': 'gloo'  # distributed
    },
    metric_definitions    = [
        {"Name": "train:loss", "Regex": r"Train Loss: ([0-9\.]+)"},
        {"Name": "train:acc",  "Regex": r"Train Acc: ([0-9\.]+)"},
        {"Name": "val:loss",   "Regex": r"Test Loss: ([0-9\.]+)"},
        {"Name": "val:acc",    "Regex": r"Test Acc: ([0-9\.]+)"},
    ],
)

estimator.fit({
    "train": TrainingInput("s3://{}/{}/dataset.zip".format(bucket, prefix)),  # /opt/ml/input/data/train
})

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2026-07-02-08-06-17-342


2026-07-02 08:06:17 Starting - Starting the training job...
2026-07-02 08:06:34 Pending - Training job waiting for capacity............
2026-07-02 08:08:37 Pending - Preparing the instances for training...
2026-07-02 08:09:08 Downloading - Downloading input data...
2026-07-02 08:09:38 Downloading - Downloading the training image........bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
CUDA compat package should be installed for NVIDIA driver smaller than 560.35.05
Current installed NVIDIA driver version is 570.211.01
Skipping CUDA compat setup as newer NVIDIA driver is installed
2026-07-02 08:11:11,104 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-07-02 08:11:11,123 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-07-02 08:11:11,133 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-07-02 

In [16]:
# Create new model + Endpoint Configuration
import boto3
import time

sm = boto3.client("sagemaker")

model = estimator.create_model(
  name = f'intel-model-{int(time.time())}',
)
model_name = model.name
model._create_sagemaker_model(instance_type="ml.g5.xlarge")

new_config = f'endpoint-configuration-{int(time.time())}'
sm.create_endpoint_config(
    EndpointConfigName=new_config,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.g5.xlarge"
    }]
)

INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-east-1-200148130345/sagemaker/output/pytorch-training-2026-07-02-08-06-17-342/output/model.tar.gz), script artifact (s3://sagemaker-us-east-1-200148130345/pytorch-training-2026-07-02-08-06-17-342/source/sourcedir.tar.gz), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-200148130345/intel-model-1782980219/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: intel-model-1782980219


{'EndpointConfigArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint-config/endpoint-configuration-1782980221',
 'ResponseMetadata': {'RequestId': 'a265bcf1-6a9a-47d4-ad19-773cbfc8a8df',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'a265bcf1-6a9a-47d4-ad19-773cbfc8a8df',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '114',
   'date': 'Thu, 02 Jul 2026 08:17:01 GMT'},
  'RetryAttempts': 0}}

In [17]:
# Create/Update Endpoint
# sm.create_endpoint(
#     EndpointName = "intel-endpoint",
#     EndpointConfigName = new_config
# )

sm.update_endpoint(
    EndpointName = "intel-endpoint",
    EndpointConfigName = new_config
)

{'EndpointArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint/intel-endpoint',
 'ResponseMetadata': {'RequestId': 'ce803d0f-6939-4199-b16e-4d39852422ab',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'ce803d0f-6939-4199-b16e-4d39852422ab',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '82',
   'date': 'Thu, 02 Jul 2026 08:17:06 GMT'},
  'RetryAttempts': 0}}

In [13]:
CLASSES = _dataset.classes
CLASSES

['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

In [19]:
# Inference test 
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'intel-endpoint'
image_path = "sea.jpg"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 4, 'confidence': 0.9779362678527832}
sea


In [20]:
# Inference test 
import base64, json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'intel-endpoint'
image_path = "forest.jpg"

with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   #           application/json | text/plain | image/png
    Body=json.dumps({"image": b64}),  # json.dumps({"image": b64}) |        b64 |  f.read()
)

body = json.load(response["Body"])
print(body)
print(CLASSES[body["class_idx"]])

{'class_idx': 1, 'confidence': 0.9999946355819702}
forest
